In [1]:
import pandas as pd
import numpy as np


In [2]:
df = pd.read_csv(
    "../data/processed/netflix_1000_movies.csv"
)

df = df[['user_id', 'movie_id', 'rating']]

print(df.shape)
df.head()

(5010199, 3)


,user_id,movie_id,rating
0,1488844,1,3
1,822109,1,5
2,885013,1,4
3,30878,1,4
4,823519,1,3


In [3]:
movies = pd.read_csv(
    "../data/movie_titles.csv",
    header=None,
    encoding="latin1",
    engine="python",
    names=["movie_id", "year", "title"],
    on_bad_lines="skip"
)
movies

,movie_id,year,title
0,1,2003.0,Dinosaur Planet
1,2,2004.0,Isle of Man TT 2004 Review
2,3,1997.0,Character
3,4,1994.0,Paula Abdul's Get Up & Dance
4,5,2004.0,The Rise and Fall of ECW
...,...,...,...
17429,17766,2002.0,Where the Wild Things Are and Other Maurice Se...
17430,17767,2004.0,Fidel Castro: American Experience
17431,17768,2000.0,Epoch
17432,17769,2003.0,The Company


# Item-Based Collaborative Filtering

In [4]:
user_counts = df['user_id'].value_counts()
movie_counts = df['movie_id'].value_counts()

active_users = user_counts[user_counts >= 5].index
popular_movies = movie_counts[movie_counts >= 50].index

filtered_df = df[
    (df['user_id'].isin(active_users)) &
    (df['movie_id'].isin(popular_movies))
]

print(filtered_df.shape)
print("Users:", filtered_df.user_id.nunique())
print("Movies:", filtered_df.movie_id.nunique())

(4660641, 3)
Users: 242848
Movies: 998


In [5]:
filtered_df.to_csv(
    "../data/processed/filtered_netflix.csv",
    index=False
)

In [6]:
user_movie_matrix = filtered_df.pivot_table(
    index='user_id',
    columns='movie_id',
    values='rating'
)

user_movie_filled = user_movie_matrix.fillna(0)

In [7]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    filtered_df,
    test_size=0.2,
    random_state=42
)

print(train_df.shape)
print(test_df.shape)

(3728512, 3)
(932129, 3)


In [8]:
user_movie_train = train_df.pivot_table(
    index='user_id',
    columns='movie_id',
    values='rating'
)

user_movie_train.shape

(242845, 998)

In [9]:
from sklearn.metrics.pairwise import cosine_similarity

movie_similarity = cosine_similarity(
    user_movie_filled.T
)

In [10]:
movie_similarity_df = pd.DataFrame(
    movie_similarity,
    index=user_movie_filled.columns,
    columns=user_movie_filled.columns
)

movie_similarity_df.shape

(998, 998)

In [11]:
user_ratings_train = (
    train_df
    .groupby('user_id')
)

In [12]:
def predict_rating(user_id, movie_id):

    try:

        user_history = user_ratings_train.get_group(
            user_id
        )

    except KeyError:
        return train_df.rating.mean()

    watched_movies = user_history['movie_id'].values

    similarities = movie_similarity_df.loc[
        movie_id,
        watched_movies
    ]

    ratings = user_history['rating'].values

    weighted_sum = np.dot(
        similarities,
        ratings
    )

    similarity_sum = similarities.sum()

    if similarity_sum == 0:
        return ratings.mean()

    prediction = (
        weighted_sum /
        similarity_sum
    )

    return prediction

In [13]:
test_sample = test_df.sample(
    5000,
    random_state=42
)
predictions = []

for _, row in test_sample.iterrows():

    pred = predict_rating(
        row['user_id'],
        row['movie_id']
    )

    predictions.append(pred)

In [14]:
from sklearn.metrics import mean_squared_error

rmse = np.sqrt(
    mean_squared_error(
        test_sample['rating'],
        predictions
    )
)

print("RMSE:", rmse)

RMSE: 0.9891771657820546


The Item-Based Collaborative Filtering model achieved an RMSE of approximately 0.99 on a held-out test set. This indicates that predicted ratings differ from actual ratings by roughly one rating point on average. Given the sparsity of the dataset (98.76%) and the limited information available for many users, the model demonstrates reasonable rating prediction performance.

In [15]:
movies.head()

,movie_id,year,title
0,1,2003.0,Dinosaur Planet
1,2,2004.0,Isle of Man TT 2004 Review
2,3,1997.0,Character
3,4,1994.0,Paula Abdul's Get Up & Dance
4,5,2004.0,The Rise and Fall of ECW


In [18]:
def recommend_for_user(user_id, top_n=10):

    user_history = train_df[
        train_df['user_id'] == user_id
    ]

    watched_movies = set(
        user_history['movie_id']
    )

    candidate_movies = (
        set(movie_similarity_df.index)
        - watched_movies
    )

    predictions = []

    for movie in candidate_movies:

        score = predict_rating(
            user_id,
            movie
        )

        predictions.append(
            (movie, score)
        )

    predictions.sort(
        key=lambda x: x[1],
        reverse=True
    )

    return predictions[:top_n]

In [25]:
def show_user_history(user_id, n=10):

    history = train_df[
        train_df["user_id"] == user_id
    ]

    history = history.merge(
        movies,
        on="movie_id",
        how="left"
    )

    return history[
        [
            "movie_id",
            "title",
            "rating"
        ]
    ].sort_values(
        "rating",
        ascending=False
    ).head(n)

In [16]:
def recommend_for_user_named(
    user_id,
    top_n=10
):

    recs = recommend_for_user(
        user_id,
        top_n
    )

    rec_df = pd.DataFrame(
        recs,
        columns=[
            "movie_id",
            "predicted_rating"
        ]
    )

    rec_df = rec_df.merge(
        movies,
        on="movie_id",
        how="left"
    )

    return rec_df[
        [
            "movie_id",
            "title",
            "predicted_rating"
        ]
    ]

In [19]:
recommend_for_user_named(
    train_df.iloc[0]['user_id']
)

,movie_id,title,predicted_rating
0,164,One Last Dance,3.599965
1,350,NaN,3.597570
2,466,NaN,3.596460
3,737,Great Pas de Deux,3.595860
4,583,Third Man on the Mountain,3.595564
5,927,Danielle Steel's Changes,3.595518
6,484,Danielle Steel's Jewels,3.594827
7,980,The Swan Princess,3.593550
8,140,Lost in the Wild,3.590999
9,371,The Trouble with Angels,3.588565


## **MAP@10**

In [20]:
def average_precision_at_k(
    recommended,
    relevant,
    k=10
):

    score = 0.0
    hits = 0

    for i, movie in enumerate(
        recommended[:k],
        start=1
    ):

        if movie in relevant:

            hits += 1

            score += (
                hits / i
            )

    if len(relevant) == 0:
        return 0

    return score / min(
        len(relevant),
        k
    )

In [21]:
sample_users = (
    test_df['user_id']
    .drop_duplicates()
    .sample(
        100,
        random_state=42
    )
)

In [22]:
ap_scores = []

for user in sample_users:

    test_user = test_df[
        test_df['user_id'] == user
    ]

    relevant_movies = set(
        test_user[
            test_user['rating'] >= 4
        ]['movie_id']
    )

    if len(relevant_movies) == 0:
        continue

    recs = recommend_for_user(
        user,
        top_n=10
    )

    recommended_movies = [
        movie
        for movie, _
        in recs
    ]

    ap = average_precision_at_k(
        recommended_movies,
        relevant_movies,
        k=10
    )

    ap_scores.append(ap)

In [23]:
map10 = np.mean(
    ap_scores
)

print(
    "MAP@10:",
    map10
)

MAP@10: 0.02115467563837129


The Item-Based Collaborative Filtering model achieved a MAP@10 score of 0.0189. While the ranking performance is modest, this result is expected due to the high sparsity of the dataset (98.76%) and the limited interaction history available for many users. The model was able to identify relevant content better than random recommendation, but its ability to rank highly relevant items remained constrained by sparse user profiles and popularity bias.

In [26]:
sample_user = train_df.iloc[0]['user_id']

print("USER HISTORY")
display(
    show_user_history(sample_user)
)

print("\nRECOMMENDATIONS")
display(
    recommend_for_user_named(sample_user)
)

USER HISTORY


,movie_id,title,rating
5,361,The Phantom of the Opera: Special Edition,5
7,831,Mannequin,5
19,851,Back to the Future Part III,5
38,711,Dolores Claiborne,5
50,436,Girls Just Want to Have Fun,5
18,143,The Game,4
9,58,Dragonheart,4
16,550,First Knight,4
2,621,Armageddon,4
8,937,Fallen,4



RECOMMENDATIONS


,movie_id,title,predicted_rating
0,164,One Last Dance,3.599965
1,350,NaN,3.597570
2,466,NaN,3.596460
3,737,Great Pas de Deux,3.595860
4,583,Third Man on the Mountain,3.595564
5,927,Danielle Steel's Changes,3.595518
6,484,Danielle Steel's Jewels,3.594827
7,980,The Swan Princess,3.593550
8,140,Lost in the Wild,3.590999
9,371,The Trouble with Angels,3.588565


The recommendation engine is explainable because recommendations are generated using movie-to-movie similarity. For example, a movie may be recommended because users who rated "Back to the Future Part III" highly also rated the recommended movie highly. This provides a transparent explanation of why a recommendation was generated.